# 02 — Technical Analyst Agent (Evening 2)

**Research question answered here:**
- **RQ3**: Can an LLM correctly classify swing setups given computed context?

**What this notebook proves:**
Kite/yfinance data → LangGraph `StateGraph` → `gpt-4.1-mini` → `TechnicalScorecard` Pydantic model → Jupyter display

**Gate before moving to Evening 3:**
- `setup_type` is correct for at least 1 stock (verified on TradingView)
- `initial_stop` is at a real swing low — not a round number
- No exceptions or schema validation errors

**Cost: ~₹40** (10-15 LLM calls at gpt-4.1-mini rates)

In [ ]:
# Cell 1 — Imports (copy data functions from 01_ inline)
import warnings; warnings.filterwarnings('ignore')
from dotenv import load_dotenv
import os, json as _json
from datetime import date
from typing import Optional, Literal

import pandas as pd
import pandas_ta as ta
import numpy as np
import yfinance as yf
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, HTML

from pydantic import BaseModel, Field, field_validator
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.graph import StateGraph, END
from typing import TypedDict

load_dotenv('../.env') or load_dotenv('.env')

AS_OF_DATE = date.today().isoformat()
print(f'as_of_date: {AS_OF_DATE}')
print(f'OpenAI key: ...{os.getenv("OPENAI_API_KEY","")[-4:]}')

In [ ]:
# Cell 2 — Data functions (copy from 01_ — will extract to src/ later)
# Spec ref: §002 data-layer plan.md

def get_ohlcv(symbol: str, as_of_date: str) -> pd.DataFrame:
    ticker = f'{symbol}.NS' if symbol != '^CNX500' else '^CNX500'
    raw = yf.download(ticker, period='max', auto_adjust=True, progress=False)
    if raw.empty:
        raise ValueError(f'No data for {ticker}')
    if isinstance(raw.columns, pd.MultiIndex):
        raw.columns = raw.columns.droplevel(1)
    raw.columns = [c.lower().replace(' ','_') for c in raw.columns]
    raw.index = pd.to_datetime(raw.index).normalize().date
    cutoff = date.fromisoformat(as_of_date)
    df = raw[raw.index <= cutoff].copy()
    if len(df) < 100:
        raise ValueError(f'{symbol} only {len(df)} rows')
    return df

def compute_indicators(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['ema_20']     = ta.ema(df['close'], length=20)
    df['ema_50']     = ta.ema(df['close'], length=50)
    df['ema_200']    = ta.ema(df['close'], length=200)
    df['atr_14']     = ta.atr(df['high'], df['low'], df['close'], length=14)
    df['rsi_14']     = ta.rsi(df['close'], length=14)
    bb = ta.bbands(df['close'], length=20, std=2.0)
    if bb is not None:
        df['bb_upper'] = bb['BBU_20_2.0']
        df['bb_lower'] = bb['BBL_20_2.0']
    df['vol_avg_20'] = df['volume'].rolling(20).mean()
    return df

def identify_swings(df: pd.DataFrame, pct: float = 0.03) -> dict:
    highs, lows = [], []
    direction, last_pivot = None, float(df['close'].iloc[0])
    for i in range(1, len(df)):
        h, l = float(df['high'].iloc[i]), float(df['low'].iloc[i])
        if direction != 'down' and h >= last_pivot * (1 + pct):
            highs.append(round(h, 2)); last_pivot = h; direction = 'down'
        elif direction != 'up' and l <= last_pivot * (1 - pct):
            lows.append(round(l, 2));  last_pivot = l; direction = 'up'
    lc = float(df['close'].iloc[-1])
    return {
        'nearest_support':    max([l for l in lows  if l < lc], default=round(float(df['low'].min()),2)),
        'nearest_resistance': min([h for h in highs if h > lc], default=round(float(df['high'].max()),2)),
        'recent_highs': sorted(highs, reverse=True)[:5],
        'recent_lows':  sorted(lows,  reverse=True)[:5],
    }

def relative_strength(symbol: str, as_of_date: str, weeks: int = 13) -> float:
    wd = weeks * 5
    s = get_ohlcv(symbol, as_of_date)
    n = get_ohlcv('^CNX500', as_of_date)
    if len(s) < wd or len(n) < wd: return 50.0
    sr = s['close'].iloc[-1] / s['close'].iloc[-wd] - 1
    nr = n['close'].iloc[-1] / n['close'].iloc[-wd] - 1
    ratio = (1 + sr) / (1 + nr)
    return round(min(100.0, max(0.0, (ratio - 0.7) / 0.6 * 100)), 1)

print('✓ Data functions loaded')

## Step 1 — Pydantic schema (TechnicalScorecard)

Spec ref: §001-models plan.md, §4 Agent 2 output schema, constitution §domain rules


In [ ]:
# Cell 3 — TechnicalScorecard Pydantic model
# Spec ref: §001-models plan.md — exact field names and types
# CRITICAL: rs_rating_pct is SEPARATE from component_scores (v1.2 bug fix)

class TechnicalScorecard(BaseModel):
    symbol:           str
    as_of_date:       str
    setup_type:       Literal['pullback_uptrend', 'breakout_base',
                              'mean_reversion_range', 'rs_leader', 'none']
    direction:        Literal['long', 'short', 'none']
    weekly_trend:     Literal['up', 'down', 'sideways']
    daily_trend:      Literal['up', 'down', 'sideways']

    # Scoring — 4 components summing to 0-100
    # v1.2 FIX: RS and RR are NOT inside component_scores
    component_scores: dict  # keys: trend(0-30), structure(0-25), setup(0-30), volume(0-15)
    technical_score:  int = Field(ge=0, le=100)

    # Separate fields — not part of technical_score
    rs_rating_pct:    float = Field(ge=0, le=100)
    rr_ratio:         float

    # Price levels
    entry_zone:       tuple[float, float]
    initial_stop:     float
    target_1:         float
    target_2:         Optional[float] = None

    # Context
    score_reasoning:  str
    volume_signal:    Literal['expansion', 'neutral', 'contraction']

    # Rejection
    rejected:         bool = False
    rejection_reason: Optional[str] = None

    @field_validator('component_scores')
    @classmethod
    def check_components(cls, v):
        required = {'trend', 'structure', 'setup', 'volume'}
        if set(v.keys()) != required:
            raise ValueError(f'component_scores must have exactly: {required}')
        return v

print('✓ TechnicalScorecard schema loaded')
print('  Fields:', list(TechnicalScorecard.model_fields.keys()))

## Step 2 — TECHNICAL_BACKSTORY (the system prompt)

This is what the LLM reads. Edit this to tune the agent's behaviour.
Each version change: bump `BACKSTORY_VERSION` and add a comment.


In [ ]:
# Cell 4 — TECHNICAL_BACKSTORY
# Spec ref: §4 Agent 2 backstory, constitution §refusal rules
# VERSION HISTORY: edit here, bump version, add comment

BACKSTORY_VERSION = 'v1.0'

TECHNICAL_BACKSTORY = """
You are a chart technician who has analysed Indian equity charts for 15 years.
You think weekly trend first, daily structure second, trigger setup third.
You believe price + volume + structure beat every indicator combination.
You use indicators only as confirmation, never as primary signal.

You know that Indian retail traders fail because they:
1. Trade against the weekly trend
2. Enter at the top of a daily move chasing momentum
3. Ignore relative strength and pick laggards
4. Use round-number stops that get hunted

YOU CLASSIFY EXACTLY FOUR SETUP TYPES — reject anything else:

1. pullback_uptrend
   Required: weekly trend UP (EMA-50 rising on weekly),
             daily price above EMA-200,
             3-7 day pullback to rising EMA-20,
             RSI reset to 40-55 zone (not oversold),
             volume below average during pullback
   Entry: at or near EMA-20 on a reversal candle
   Stop:  below the pullback swing low minus 0.3x ATR

2. breakout_base
   Required: 6-12 week consolidation visible on daily chart,
             ATR declining during base (contraction),
             clear horizontal resistance identified
   Trigger: close above resistance with volume >= 1.5x 20-day average
   Entry:   at or near the breakout close
   Stop:    below base midpoint

3. mean_reversion_range
   Required: horizontal range clearly visible for 4+ weeks,
             price at range support (long) or resistance (short),
             RSI not in strong trend territory
   Entry:   at range boundary with reversal candle
   Stop:    beyond range extreme by 1x ATR

4. rs_leader
   Required: RS rating above 75th percentile vs Nifty500 (13-week),
             one of the above 3 setups also present,
             sector RS in top 30%
   This is an overlay — high RS amplifies conviction in the base setup

SCORING (4 components, total 0-100):
- trend    (0-30): weekly and daily EMA alignment with setup direction
- structure (0-25): quality of swing levels, support/resistance clarity
- setup    (0-30): precision of match to one of the 4 templates above
- volume   (0-15): volume confirms the setup story

REFUSAL RULES — apply strictly:
- Reject if fewer than 100 trading days of history available
- Reject if weekly trend does not agree with the setup direction
- Reject if RSI is above 75 at entry for a long setup
- Reject if stop distance would exceed 8% of entry price
- Reject if R:R ratio is below 1.8
- Reject if the chart is choppy with no identifiable structure
- Reject if price is below EMA-200 for a long setup
- DO NOT invent a setup — set rejected=True with a clear reason

STOP PLACEMENT RULE:
- Stops must be placed at real structural swing lows/highs from the data
- NEVER place stops at round numbers (e.g. 1500, 2000, 1450)
- The stop level must correspond to a real price in the provided data
"""

print(f'✓ TECHNICAL_BACKSTORY loaded  (version: {BACKSTORY_VERSION})')
print(f'  Length: {len(TECHNICAL_BACKSTORY)} chars')

In [ ]:
# Cell 5 — build_prompt: format indicator context for LLM
# LLM receives structured text — all arithmetic is done in Python, not by LLM

def build_prompt(symbol: str, as_of_date: str, df: pd.DataFrame,
                 swings: dict, rs: float) -> str:
    """Format indicator context as a structured text prompt for the LLM."""
    last = df.iloc[-1]
    prev = df.iloc[-2]

    # Weekly trend proxy: slope of EMA-200 over last 10 bars
    ema200_slope = (float(df['ema_200'].iloc[-1]) - float(df['ema_200'].iloc[-10]))
    weekly_trend_proxy = 'rising' if ema200_slope > 0 else 'flat/falling'

    # Volume signal
    vol_ratio = float(last['volume']) / float(last['vol_avg_20'])
    vol_signal = 'expansion' if vol_ratio > 1.3 else ('contraction' if vol_ratio < 0.7 else 'neutral')

    return f"""Analyse {symbol} as of {as_of_date} for a potential swing trade setup.

PRICE & TREND:
  Close:       ₹{float(last['close']):.2f}
  EMA-20:      ₹{float(last['ema_20']):.2f}  ({'ABOVE' if last['close'] > last['ema_20'] else 'BELOW'})
  EMA-50:      ₹{float(last['ema_50']):.2f}  ({'ABOVE' if last['close'] > last['ema_50'] else 'BELOW'})
  EMA-200:     ₹{float(last['ema_200']):.2f} ({'ABOVE' if last['close'] > last['ema_200'] else 'BELOW'})
  EMA-200 slope (10-bar): {weekly_trend_proxy}

VOLATILITY & MOMENTUM:
  ATR(14):     ₹{float(last['atr_14']):.2f} ({float(last['atr_14'])/float(last['close'])*100:.1f}% of price)
  RSI(14):     {float(last['rsi_14']):.1f}
  BB Upper:    ₹{float(last.get('bb_upper', 0)):.2f}
  BB Lower:    ₹{float(last.get('bb_lower', 0)):.2f}

VOLUME:
  Today:       {float(last['volume']):,.0f}  ({vol_ratio*100:.0f}% of 20-day average)
  Signal:      {vol_signal}

SWING LEVELS (from ZigZag 3%):
  Nearest support:    ₹{swings['nearest_support']}
  Nearest resistance: ₹{swings['nearest_resistance']}
  Recent swing highs: {swings['recent_highs']}
  Recent swing lows:  {swings['recent_lows']}

RELATIVE STRENGTH:
  RS vs Nifty500 (13-week): {rs:.1f}th percentile

Classify the swing setup using the four templates.
Place the stop at the nearest structural swing low from the data above.
Do not use round numbers for stops.
If no setup qualifies, set rejected=True with a clear rejection_reason.
"""

# Test the prompt
df_test = get_ohlcv('INFY', AS_OF_DATE)
df_test = compute_indicators(df_test)
sw_test = identify_swings(df_test)
rs_test = relative_strength('INFY', AS_OF_DATE)

prompt = build_prompt('INFY', AS_OF_DATE, df_test, sw_test, rs_test)
print(prompt)

## Step 3 — LangGraph node + LLM call


In [ ]:
# Cell 6 — PipelineState and technical_node
# Spec ref: §5.1 PipelineState, §5.3 node function pattern

class PipelineState(TypedDict):
    run_date:   str
    symbol:     str
    scorecard:  Optional[TechnicalScorecard]
    errors:     list
    cost_usd:   float


def technical_node(state: PipelineState) -> PipelineState:
    """
    LangGraph node: fetch data → compute indicators → call LLM → return TechnicalScorecard.
    Never raises — errors go to state['errors'].
    Spec ref: §5.3 Node function pattern
    """
    symbol    = state['symbol']
    as_of     = state['run_date']

    try:
        # Pure Python — no LLM for data or indicators
        df     = get_ohlcv(symbol, as_of)
        df     = compute_indicators(df)
        swings = identify_swings(df)
        rs     = relative_strength(symbol, as_of)

        # Build prompt
        prompt = build_prompt(symbol, as_of, df, swings, rs)

        # LLM call — gpt-4.1-mini, temp=0, seed=42 (spec §3.3)
        llm = ChatOpenAI(model='gpt-4.1-mini', temperature=0, seed=42)
        structured_llm = llm.with_structured_output(TechnicalScorecard)

        result = structured_llm.invoke([
            SystemMessage(content=TECHNICAL_BACKSTORY),
            HumanMessage(content=prompt),
        ])

        # Override rs_rating_pct with Python-computed value (not LLM estimate)
        result_dict = result.model_dump()
        result_dict['rs_rating_pct'] = rs
        result = TechnicalScorecard(**result_dict)

        return {**state, 'scorecard': result}

    except Exception as e:
        error_msg = f'technical_node/{symbol}: {type(e).__name__}: {e}'
        print(f'  ✗ {error_msg}')
        return {**state, 'errors': state['errors'] + [error_msg]}


print('✓ PipelineState and technical_node defined')

In [ ]:
# Cell 7 — Wire into LangGraph StateGraph
# Spec ref: §5.2 StateGraph node wiring

def build_single_agent_graph():
    """Minimal StateGraph: one node, one stock. The POC graph."""
    graph = StateGraph(PipelineState)
    graph.add_node('technical', technical_node)
    graph.set_entry_point('technical')
    graph.add_edge('technical', END)
    return graph.compile()


app = build_single_agent_graph()
print('✓ LangGraph compiled')
print('  Nodes:', list(app.get_graph().nodes.keys()))

In [ ]:
# Cell 8 — RUN on INFY  (~₹5 LLM cost)
print(f'Running Technical Analyst on INFY ({AS_OF_DATE})...\n')

initial_state = {
    'run_date': AS_OF_DATE,
    'symbol':   'INFY',
    'scorecard': None,
    'errors':    [],
    'cost_usd':  0.0,
}

result = app.invoke(initial_state)

if result['errors']:
    print('ERRORS:', result['errors'])
elif result['scorecard']:
    sc = result['scorecard']
    print(f'setup_type:      {sc.setup_type}')
    print(f'direction:       {sc.direction}')
    print(f'technical_score: {sc.technical_score}/100')
    print(f'components:      {sc.component_scores}')
    print(f'rs_rating_pct:   {sc.rs_rating_pct:.1f}  (Python-computed, not LLM)')
    print(f'rr_ratio:        {sc.rr_ratio:.1f}x')
    print()
    if not sc.rejected:
        print(f'entry_zone:      ₹{sc.entry_zone[0]:.2f} – ₹{sc.entry_zone[1]:.2f}')
        print(f'initial_stop:    ₹{sc.initial_stop:.2f}')
        print(f'target_1:        ₹{sc.target_1:.2f}')
        if sc.target_2: print(f'target_2:        ₹{sc.target_2:.2f}')
    else:
        print(f'REJECTED: {sc.rejection_reason}')
    print()
    print(f'reasoning: {sc.score_reasoning}')

## Step 4 — Evaluate output (the manual check)

Compare LLM output against the actual TradingView chart.


In [ ]:
# Cell 9 — evaluate_stock: chart + scorecard + TradingView check prompts
# Spec ref: §19.1 standard evaluation cell

def evaluate_stock(symbol: str, as_of_date: str, scorecard: TechnicalScorecard) -> None:
    """Render chart with agent levels + scorecard table + manual check prompts."""
    df = get_ohlcv(symbol, as_of_date)
    df = compute_indicators(df)
    df_plot = df.iloc[-60:]
    x = [str(d) for d in df_plot.index]

    fig = go.Figure()
    fig.add_trace(go.Candlestick(
        x=x, open=df_plot['open'], high=df_plot['high'],
        low=df_plot['low'],  close=df_plot['close'],
        name='OHLC',
        increasing_line_color='#26a69a',
        decreasing_line_color='#ef5350',
    ))
    for col, color, name in [
        ('ema_20', '#FF9800', 'EMA-20'),
        ('ema_50', '#2196F3', 'EMA-50'),
    ]:
        fig.add_trace(go.Scatter(x=x, y=df_plot[col], name=name,
                                 line=dict(color=color, width=1.5)))

    # Agent levels as horizontal lines
    if not scorecard.rejected:
        fig.add_hline(y=scorecard.entry_zone[0], line_color='#4CAF50',
                      line_dash='dash', annotation_text='Entry low')
        fig.add_hline(y=scorecard.entry_zone[1], line_color='#4CAF50',
                      line_dash='dash', annotation_text='Entry high')
        fig.add_hline(y=scorecard.initial_stop,  line_color='#F44336',
                      line_dash='solid',
                      annotation_text=f'Stop ₹{scorecard.initial_stop:.0f}')
        fig.add_hline(y=scorecard.target_1,      line_color='#2196F3',
                      line_dash='dot',
                      annotation_text=f'T1 ₹{scorecard.target_1:.0f}')
        if scorecard.target_2:
            fig.add_hline(y=scorecard.target_2, line_color='#9C27B0',
                          line_dash='dot',
                          annotation_text=f'T2 ₹{scorecard.target_2:.0f}')

    setup_label = scorecard.setup_type if not scorecard.rejected else f'REJECTED: {scorecard.rejection_reason[:40]}'
    fig.update_layout(
        title=f'{symbol}  |  {as_of_date}  |  {setup_label}  |  score: {scorecard.technical_score}',
        height=440, xaxis_rangeslider_visible=False,
        template='plotly_dark',
        legend=dict(orientation='h', yanchor='bottom', y=1.01),
    )
    fig.show()

    # Scorecard HTML table
    if not scorecard.rejected:
        rr   = f'{scorecard.rr_ratio:.1f}x'
        stop_pct = abs(scorecard.initial_stop - scorecard.entry_zone[0]) / scorecard.entry_zone[0] * 100
        html = f"""
        <table style="font-size:13px;border-collapse:collapse;width:600px;margin-top:10px">
          <tr><td style="padding:5px 12px;background:#1e1e2e;color:#cdd6f4;font-weight:600;width:180px">Setup</td>
              <td style="padding:5px 12px;background:#181825;color:#a6e3a1">{scorecard.setup_type}</td></tr>
          <tr><td style="padding:5px 12px;background:#1e1e2e;color:#cdd6f4;font-weight:600">Score</td>
              <td style="padding:5px 12px;background:#181825;color:#cdd6f4">{scorecard.technical_score}/100  —  {scorecard.component_scores}</td></tr>
          <tr><td style="padding:5px 12px;background:#1e1e2e;color:#cdd6f4;font-weight:600">Entry zone</td>
              <td style="padding:5px 12px;background:#181825;color:#cdd6f4">₹{scorecard.entry_zone[0]:.2f} – ₹{scorecard.entry_zone[1]:.2f}</td></tr>
          <tr><td style="padding:5px 12px;background:#1e1e2e;color:#cdd6f4;font-weight:600">Stop loss</td>
              <td style="padding:5px 12px;background:#181825;color:#f38ba8">₹{scorecard.initial_stop:.2f}  (-{stop_pct:.1f}%)</td></tr>
          <tr><td style="padding:5px 12px;background:#1e1e2e;color:#cdd6f4;font-weight:600">Target 1 / R:R</td>
              <td style="padding:5px 12px;background:#181825;color:#a6e3a1">₹{scorecard.target_1:.2f}  /  {rr}</td></tr>
          <tr><td style="padding:5px 12px;background:#1e1e2e;color:#cdd6f4;font-weight:600">RS rating</td>
              <td style="padding:5px 12px;background:#181825;color:#cdd6f4">{scorecard.rs_rating_pct:.0f}th percentile vs Nifty500</td></tr>
          <tr><td style="padding:5px 12px;background:#1e1e2e;color:#cdd6f4;font-weight:600">Reasoning</td>
              <td style="padding:5px 12px;background:#181825;color:#cdd6f4;font-style:italic">{scorecard.score_reasoning}</td></tr>
        </table>
        """
    else:
        html = f'<div style="background:#1e1e2e;color:#f38ba8;padding:10px;border-radius:6px">REJECTED: {scorecard.rejection_reason}</div>'

    display(HTML(html))

    # Manual check prompts
    tv_url = f'https://www.tradingview.com/chart/?symbol=NSE:{symbol}'
    display(HTML(f"""
    <div style="margin-top:12px;padding:10px 14px;background:#1e1e2e;
                border-left:3px solid #f9a825;font-size:12px;color:#cdd6f4">
      <b style="color:#f9a825">Manual check — open TradingView and verify:</b>
      <a href="{tv_url}" target="_blank" style="color:#89b4fa;margin-left:10px">Open NSE:{symbol} ↗</a><br><br>
      ☐ &nbsp;Setup type <b>{scorecard.setup_type}</b> matches what you see on the chart?<br>
      ☐ &nbsp;Stop ₹{scorecard.initial_stop:.2f} is at a real swing low — not a round number?<br>
      ☐ &nbsp;Entry zone ₹{scorecard.entry_zone[0]:.0f}–{scorecard.entry_zone[1]:.0f} is near the current price?<br>
      ☐ &nbsp;Weekly trend <b>{scorecard.weekly_trend}</b> agrees with the trade direction?<br>
      ☐ &nbsp;R:R {scorecard.rr_ratio:.1f}x uses a real structural target — not an arbitrary price?<br>
    </div>
    """))


# Display the INFY result from Cell 8
if result.get('scorecard'):
    evaluate_stock('INFY', AS_OF_DATE, result['scorecard'])

In [ ]:
# Cell 10 — Run on all 5 POC stocks  (~₹35 LLM cost)
POC_STOCKS = ['INFY', 'RELIANCE', 'DRREDDY', 'LT', 'HDFCBANK']

print(f'Running Technical Analyst on {len(POC_STOCKS)} stocks...\n')
scorecards = {}

for sym in POC_STOCKS:
    state = {'run_date': AS_OF_DATE, 'symbol': sym,
             'scorecard': None, 'errors': [], 'cost_usd': 0.0}
    out = app.invoke(state)
    if out['scorecard']:
        sc = out['scorecard']
        scorecards[sym] = sc
        status = f'{sc.setup_type} ({sc.technical_score})' if not sc.rejected else f'rejected: {sc.rejection_reason[:50]}'
        print(f'  {sym:<12} {status}')
    else:
        print(f'  {sym:<12} ERROR: {out["errors"]}')

print(f'\nCandidates: {sum(1 for sc in scorecards.values() if not sc.rejected)}/{len(POC_STOCKS)}')

In [ ]:
# Cell 11 — Summary table
rows = []
for sym, sc in scorecards.items():
    if not sc.rejected:
        rows.append({
            'Symbol':  sym,
            'Setup':   sc.setup_type,
            'Score':   sc.technical_score,
            'RS':      f'{sc.rs_rating_pct:.0f}th',
            'Entry low': f'₹{sc.entry_zone[0]:.0f}',
            'Stop':    f'₹{sc.initial_stop:.0f}',
            'T1':      f'₹{sc.target_1:.0f}',
            'R:R':     f'{sc.rr_ratio:.1f}x',
        })
    else:
        rows.append({
            'Symbol': sym, 'Setup': '— rejected —',
            'Score': 0,    'RS': '—',
            'Entry low': '—', 'Stop': '—', 'T1': '—', 'R:R': '—',
        })

df_summary = pd.DataFrame(rows)
display(df_summary.style.apply(
    lambda r: ['background-color:#1a1a2e;color:#a6e3a1' if r['Setup'] != '— rejected —'
               else 'background-color:#1a1a2e;color:#585b70'] * len(r), axis=1
))

In [ ]:
# Cell 12 — Evaluate each non-rejected candidate
for sym, sc in scorecards.items():
    if not sc.rejected:
        print(f'\n{"-"*60}')
        evaluate_stock(sym, AS_OF_DATE, sc)

## Backstory tuning loop

If a stock is classified incorrectly:

1. Note which check failed (wrong setup type, or stop on round number)
2. Edit `TECHNICAL_BACKSTORY` in Cell 4
3. Re-run Cell 4 to update the string in memory
4. Re-run Cell 8 or 10 to test the change
5. Repeat until the output looks right on TradingView

Bump `BACKSTORY_VERSION` each time (v1.0 → v1.1 → v1.2)

**Evening 2 Gate:**
- [ ] At least 1 candidate has correct setup_type (verified on TradingView)
- [ ] Stop is at a real swing low — not a round number
- [ ] No Python exceptions or schema validation errors
- [ ] LangSmith trace shows full prompt + response (if LANGCHAIN_API_KEY set)